# Airline Passenger Segmentation using K-Means Clustering

**Goal:** group airline passengers into segments with similar service-rating patterns, so an airline can see which groups are happy, which are not, and why.

**Dataset:** Invistico Airline passenger satisfaction data (129,880 passengers, 23 columns).

**Pipeline:** load data → select the 14 service-rating columns as `X` → scale (z-score) → choose K with the elbow method → K-Means → profile the clusters → save the model → Streamlit app (`app.py`)

**Files in the same folder as this notebook**

| File | Purpose |
|---|---|
| `Invistico_Airline.csv` | Dataset (input) |
| `utils.py` | Shared helper code (created in section 0) |
| `model/` | Saved scaler, K-Means, PCA, profile tables (created in section 9) |
| `sample_data.csv` | 2,000 passengers for the app demo (created in section 9) |
| `app.py` | Streamlit app (created in section 11) |

## 0. Helper file

This cell writes `utils.py`. The notebook and the Streamlit app both use it, so the features are selected in exactly the same way in training and in the app.

In [ ]:
%%writefile utils.py
"""Shared helpers for the Airline Passenger Segmentation (K-Means) project.

Used by both the notebook (training) and app.py (Streamlit), so the feature
selection is identical in both places.
"""
import numpy as np
import pandas as pd

TARGET_COL = "satisfaction"  # NOT used for clustering, only to describe the clusters

# The 14 service-rating columns (0 to 5) used as X for K-Means
RATING_COLS = [
    "Seat comfort",
    "Departure/Arrival time convenient",
    "Food and drink",
    "Gate location",
    "Inflight wifi service",
    "Inflight entertainment",
    "Online support",
    "Ease of Online booking",
    "On-board service",
    "Leg room service",
    "Baggage handling",
    "Checkin service",
    "Cleanliness",
    "Online boarding",
]

# Optional columns: used only to describe the clusters when they are present
PROFILE_COLS = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
]


def prepare_features(df):
    """Return the 14 rating columns as the feature matrix X for K-Means."""
    missing = [c for c in RATING_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required rating columns: {missing}")
    X = df[RATING_COLS].apply(pd.to_numeric, errors="coerce")
    bad = [c for c in RATING_COLS if X[c].isna().any()]
    if bad:
        raise ValueError(f"Missing or non-numeric values in columns: {bad}")
    return X.astype(float)


def assign_clusters(df, scaler, kmeans):
    """Return (cluster labels, scaled feature matrix) for raw passenger rows."""
    X_scaled = scaler.transform(prepare_features(df))
    return kmeans.predict(X_scaled), X_scaled


def find_elbow(ks, inertias):
    """Elbow of the inertia curve: the point farthest from the straight line
    joining the first and the last point (same idea as KneeLocator)."""
    ks = np.asarray(list(ks), dtype=float)
    y = np.asarray(inertias, dtype=float)
    x_n = (ks - ks.min()) / (ks.max() - ks.min())
    y_n = (y - y.min()) / (y.max() - y.min())
    # line from (0, y_n[0]) to (1, y_n[-1]); distance of each point to that line
    dist = np.abs((y_n[-1] - y_n[0]) * x_n - (1.0) * (y_n - y_n[0])) / np.hypot(y_n[-1] - y_n[0], 1.0)
    return int(ks[int(np.argmax(dist))])


def cluster_traits(centers, top=3, cutoff=0.25):
    """Describe each cluster by the ratings that differ most from the average.

    `centers` is a DataFrame (clusters x RATING_COLS) of standardized centroids.
    """
    traits = {}
    for c in centers.index:
        z = centers.loc[c].sort_values(ascending=False)
        high = [f for f in z.index[:top] if z[f] > cutoff]
        low = [f for f in z.index[::-1][:top] if z[f] < -cutoff]
        parts = []
        if high:
            parts.append("Rated higher: " + ", ".join(high))
        if low:
            parts.append("Rated lower: " + ", ".join(low))
        traits[str(c)] = " | ".join(parts) if parts else "Close to the average passenger"
    return traits

## 1. Setup and load the data

In [ ]:
# Optional: pip install kneed   (elbow detection used in class; a built-in fallback is used if it is missing)
import json
import os
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

from utils import RATING_COLS, TARGET_COL, assign_clusters, cluster_traits, find_elbow, prepare_features

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
DATA_DIR = os.getcwd()   # or e.g. DATA_DIR = r"C:\data" if the CSV is somewhere else
CSV_PATH = os.path.join(DATA_DIR, "Invistico_Airline.csv")
MODEL_DIR = os.path.join(DATA_DIR, "model")
os.makedirs(MODEL_DIR, exist_ok=True)

airlines = pd.read_csv(CSV_PATH)
airlines.shape

In [ ]:
airlines.head()

In [ ]:
airlines.columns

In [ ]:
airlines.satisfaction.value_counts()

In [ ]:
print("Missing values per column (only columns with missing values):")
print(airlines.isna().sum()[airlines.isna().sum() > 0])
print("Duplicate rows:", airlines.duplicated().sum())

## 2. Select X (features for K-Means)

`X` is the 14 service-rating columns, from `Seat comfort` to `Online boarding`. Each rating is 0 to 5, where 0 means the service was not applicable.

`satisfaction` is **not** part of `X`. K-Means is unsupervised, so we only use satisfaction afterwards to describe the clusters.

In [ ]:
X = airlines.loc[:, 'Seat comfort':'Online boarding']
assert list(X.columns) == RATING_COLS   # same columns that utils.py uses
print(X.shape)
X.describe()

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for ax, col in zip(axes.ravel(), RATING_COLS):
    X[col].value_counts().sort_index().plot(kind="bar", ax=ax)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("")
for ax in axes.ravel()[len(RATING_COLS):]:
    ax.axis("off")
plt.suptitle("Distribution of the 14 ratings", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(X.corr(), cmap="coolwarm", center=0)
plt.title("Correlation between the ratings")
plt.show()

## 3. Scaling: Min-Max vs Z-score (Standard) scaling

K-Means works with **distances**. If one feature has big numbers (for example flight distance in thousands) and another has small numbers (ratings 0 to 5), the big one dominates the distance. Scaling puts all features on a comparable scale.

| | Min-Max scaling | Z-score / Standard scaling |
|---|---|---|
| Formula | `(x - min) / (max - min)` | `(x - mean) / std` |
| Result | values between **0 and 1** | **mean = 0, std = 1** (can be negative) |
| Meaning | position between the smallest and the largest value | how many standard deviations above or below the mean |
| sklearn | `MinMaxScaler` | `StandardScaler` |
| Weakness | one big outlier squeezes all other values together | assumes a roughly bell-shaped spread |

**Example (marks of 10 students)**

In [ ]:
marks = np.array([45, 78, 82, 70, 90, 88, 72, 87, 60, 85])

minmax = (marks - marks.min()) / (marks.max() - marks.min())
zscore = (marks - marks.mean()) / marks.std()

example = pd.DataFrame({"marks": marks, "min-max": minmax.round(4), "z-score": zscore.round(4)})
display(example)
print("Mean of marks:", marks.mean())
print("Min-max   -> min:", minmax.min(), " max:", minmax.max())
print("Z-score   -> mean:", round(zscore.mean(), 6), " std:", round(zscore.std(), 6))

How to read it:

- **45** is the smallest mark, so min-max gives **0**. **90** is the largest, so it gives **1**.
- The mean is **75.7**. A mark of 78 is slightly above it, so its z-score is a small positive number (about 0.17). A mark of 45 is far below the mean, so its z-score is very negative (about -2.26).
- Mark **below the mean = negative z-score**, **above the mean = positive**, **equal to the mean = 0**.

`StandardScaler` does the same z-score calculation. sklearn expects a 2D array (rows = samples, columns = features), so one column of marks must be reshaped to `(10, 1)`:

In [ ]:
print(StandardScaler().fit_transform(marks.reshape(-1, 1)).ravel().round(4))   # same as the z-score above
print(MinMaxScaler().fit_transform(marks.reshape(-1, 1)).ravel().round(4))     # same as the min-max above

# If you pass the 10 marks as ONE row (1 sample, 10 features), each feature has std = 0, so every value becomes 0:
print(StandardScaler().fit_transform(marks.reshape(1, -1)))

**What we use here:** `StandardScaler` (z-score). After scaling, every rating has mean 0 and standard deviation 1, so no rating counts more than another in the distance calculation.

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
X_arr = X_scaled.to_numpy()   # plain array used for K-Means and PCA
X_scaled.describe().round(3)

## 4. Choose the number of clusters (K)

- **Elbow method:** run K-Means for K = 1 to 10 and plot the inertia (sum of squared distances to the cluster center). The "elbow" is where adding more clusters stops helping much.
- **Silhouette score:** how well each passenger fits its own cluster compared with the nearest other cluster (from -1 to 1, higher is better). It is calculated on a sample of 10,000 passengers to keep it fast.

In [ ]:
ks = list(range(1, 11))
inertias = []
for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    model.fit(X_arr)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(ks, inertias, marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow plot")
plt.show()

In [ ]:
elbow_k = None
try:
    from kneed import KneeLocator
    kl = KneeLocator(ks, inertias, curve="convex", direction="decreasing")
    elbow_k = kl.elbow
    print("Elbow from KneeLocator:", elbow_k)
except ImportError:
    print("kneed is not installed, using the built-in elbow finder")
if not elbow_k:
    elbow_k = find_elbow(ks, inertias)
elbow_k = int(elbow_k)
print("Chosen K:", elbow_k)

In [ ]:
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_arr), size=10000, replace=False)
sil_scores = {}
for k in range(2, 11):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X_arr)
    sil_scores[k] = silhouette_score(X_arr[sample_idx], labels_k[sample_idx])

k_table = pd.DataFrame({"k": ks, "inertia": inertias})
k_table["silhouette"] = k_table["k"].map(sil_scores)
display(k_table.round(4))

plt.figure(figsize=(7, 4))
plt.plot(list(sil_scores.keys()), list(sil_scores.values()), marker="o")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette score")
plt.title("Silhouette score")
plt.show()

**Reading the two plots:** the inertia curve bends at the chosen K (the elbow), so that is the number we use. The silhouette scores are low overall (about 0.15 to 0.20), which is common for rating data: the groups overlap and have no sharp boundary. They are still useful for describing different passenger types, as the profile in section 7 shows.

## 5. Fit the final K-Means model

In [ ]:
K = elbow_k   # change this number if you want to try another K

kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
kmeans.fit(X_arr)
clusters = kmeans.labels_

airlines["cluster"] = clusters
np.unique(clusters, return_counts=True)

In [ ]:
centers = kmeans.cluster_centers_
print("Cluster centers shape:", centers.shape)

centers_df = pd.DataFrame(centers, columns=RATING_COLS)
centers_df.index.name = "cluster"
centers_df.round(2)

In [ ]:
plt.figure(figsize=(12, 3 + 0.5 * K))
sns.heatmap(centers_df, annot=True, fmt=".1f", cmap="RdBu_r", center=0)
plt.title("Cluster centers (standardized ratings): positive = above average, negative = below average")
plt.show()

## 6. Visualise the clusters in 2D (PCA)

PCA squeezes the 14 ratings into 2 components so the clusters can be drawn on a plane. Some overlap is expected.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE).fit(X_arr)
print("Variance explained by the 2 components:", pca.explained_variance_ratio_.round(3), "=", round(pca.explained_variance_ratio_.sum(), 3))

viz_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_arr), size=8000, replace=False)
coords = pca.transform(X_arr[viz_idx])
plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=clusters[viz_idx], palette="tab10", s=12, alpha=0.6)
plt.xlabel("PCA component 1")
plt.ylabel("PCA component 2")
plt.title("Passenger clusters")
plt.legend(title="Cluster")
plt.show()

## 7. Profile the clusters

We look at the average rating of each cluster, and then at who is in it. `satisfaction`, class, type of travel, age and delays were **not** used for clustering, so they tell us whether the clusters are meaningful.

In [ ]:
airlines.groupby("cluster")[RATING_COLS].mean().round(2).T

### 7a. Cluster centers as bar charts

Each bar is the standardized rating of one service in that cluster. A bar **above 0** means the cluster rates that service higher than the average passenger, **below 0** means lower.

In [ ]:
fig, axes = plt.subplots(K, 1, figsize=(12, 3.2 * K), sharex=True)
for c, ax in zip(range(K), np.atleast_1d(axes)):
    ax.bar(X.columns, centers_df.iloc[c])
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Cluster {c}: standardized ratings (cluster center)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

### 7b. Look at the passengers inside each cluster

`airlines[clusters == c]` gives the rows that belong to cluster `c`. Change the number in the last line to look at another cluster.

In [ ]:
cluster_dfs = {c: airlines[clusters == c] for c in range(K)}
for c, d in cluster_dfs.items():
    print(f"Cluster {c}: {len(d)} passengers")

cluster_dfs[0].head()

### 7c. Age of the passengers in each cluster

In [ ]:
airlines.groupby("cluster")["Age"].describe().round(1)

### 7d. Satisfaction and travel class inside each cluster

The first table counts passengers. The second shows the share of satisfied passengers for every class inside each cluster.

In [ ]:
for c, d in cluster_dfs.items():
    print(f"--- Cluster {c} ---")
    print(d.groupby(["satisfaction", "Class"]).size())
    print()

In [ ]:
(airlines.groupby(["cluster", "Class"])["satisfaction"]
         .apply(lambda s: s.eq("satisfied").mean())
         .unstack().round(2))

### 7e. Satisfaction and gender inside each cluster

In [ ]:
for c, d in cluster_dfs.items():
    print(f"--- Cluster {c} ---")
    print(d.groupby(["satisfaction", "Gender"]).size())
    print()

In [ ]:
(airlines.groupby(["cluster", "Gender"])["satisfaction"]
         .apply(lambda s: s.eq("satisfied").mean())
         .unstack().round(2))

### 7f. Silhouette score of the final model

Calculated on a random sample of 10,000 passengers (the full data would be slow). It is close to 0.16, so the clusters overlap.

In [ ]:
final_sil = silhouette_score(X_arr[sample_idx], clusters[sample_idx])
print("Silhouette score of the final model:", round(final_sil, 4))

### 7g. Summary table and mix of each cluster

In [ ]:
summary = pd.DataFrame({
    "passengers": airlines["cluster"].value_counts().sort_index(),
})
summary["share %"] = (summary["passengers"] / len(airlines) * 100).round(1)
summary["satisfied %"] = (airlines[TARGET_COL].eq("satisfied").groupby(airlines["cluster"]).mean() * 100).round(1)
summary["avg age"] = airlines.groupby("cluster")["Age"].mean().round(1)
summary["avg flight distance"] = airlines.groupby("cluster")["Flight Distance"].mean().round(0)
summary["avg arrival delay (min)"] = airlines.groupby("cluster")["Arrival Delay in Minutes"].mean().round(1)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["Class", "Type of Travel", "Customer Type"]):
    pd.crosstab(airlines["cluster"], airlines[col], normalize="index").plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(col + " mix per cluster")
    ax.set_ylabel("share")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
traits = cluster_traits(centers_df)
for c, text in traits.items():
    print(f"Cluster {c}: {text}")

**How to read the profile:** compare the `satisfied %` column with the rating tables. Clusters whose passengers rate most services high have a high satisfied share, and clusters with low ratings for particular services (for example online services or seat comfort) show where the airline loses passengers.

## 8. Does the choice of scaler change the result?

We repeat the clustering with Min-Max scaling and compare it with the Z-score result using the **Adjusted Rand Index** (1 = identical grouping, about 0 = unrelated). All 14 ratings are already on the same 0 to 5 scale, so the two scalers should give similar clusters here. Scaling matters much more when features have very different units (for example age, distance and ratings together).

In [ ]:
minmax_labels = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE).fit_predict(MinMaxScaler().fit_transform(X))
print("Adjusted Rand Index (Min-Max vs Z-score clusters):", round(adjusted_rand_score(clusters, minmax_labels), 3))

## 9. Save the model and files for the app

In [ ]:
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))
joblib.dump(kmeans, os.path.join(MODEL_DIR, "kmeans.joblib"))
joblib.dump(pca, os.path.join(MODEL_DIR, "pca.joblib"))

# cluster profile table (used by the app)
profile_df = airlines.groupby("cluster")[RATING_COLS].mean().round(2)
profile_df.insert(0, "size", airlines["cluster"].value_counts().sort_index())
profile_df.insert(1, "share_pct", (profile_df["size"] / len(airlines) * 100).round(1))
profile_df.insert(2, "satisfied_pct", (airlines[TARGET_COL].eq("satisfied").groupby(airlines["cluster"]).mean() * 100).round(1))
profile_df["avg_age"] = airlines.groupby("cluster")["Age"].mean().round(1)
profile_df["pct_business_class"] = (airlines["Class"].eq("Business").groupby(airlines["cluster"]).mean() * 100).round(1)
profile_df["pct_business_travel"] = (airlines["Type of Travel"].eq("Business travel").groupby(airlines["cluster"]).mean() * 100).round(1)
profile_df["pct_loyal"] = (airlines["Customer Type"].eq("Loyal Customer").groupby(airlines["cluster"]).mean() * 100).round(1)
profile_df.reset_index().to_csv(os.path.join(MODEL_DIR, "cluster_profile.csv"), index=False)

k_table.to_csv(os.path.join(MODEL_DIR, "k_selection.csv"), index=False)

meta = {
    "k": int(K),
    "features": RATING_COLS,
    "traits": traits,
    "n_passengers": int(len(airlines)),
    "inertia": float(kmeans.inertia_),
}
with open(os.path.join(MODEL_DIR, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, indent=2)

# small sample for the app demo (original columns only)
airlines.drop(columns="cluster").sample(2000, random_state=RANDOM_STATE).to_csv(
    os.path.join(DATA_DIR, "sample_data.csv"), index=False)

print("Saved:", sorted(os.listdir(MODEL_DIR)), "+ sample_data.csv")

## 10. Test the saved model on new rows

In [ ]:
scaler_loaded = joblib.load(os.path.join(MODEL_DIR, "scaler.joblib"))
kmeans_loaded = joblib.load(os.path.join(MODEL_DIR, "kmeans.joblib"))

test_rows = pd.read_csv(os.path.join(DATA_DIR, "sample_data.csv")).head(5)
labels_new, _ = assign_clusters(test_rows, scaler_loaded, kmeans_loaded)
pd.DataFrame({"satisfaction": test_rows[TARGET_COL], "predicted cluster": labels_new})

## 11. Streamlit app

The next cell writes `app.py`. It loads the saved model files and offers three tabs:

1. **Segment a CSV:** use the sample data or upload your own passengers, see cluster sizes, a 2D picture, average ratings per cluster, and download the result.
2. **Single passenger:** enter the 14 ratings and see which cluster the passenger belongs to.
3. **Model details:** elbow and silhouette plots, cluster profile and cluster centers.

In [ ]:
%%writefile app.py
"""
Airline Passenger Segmentation - Streamlit app (K-Means on service ratings).

Run:  streamlit run app.py
Needs the files created by the notebook: the model/ folder and sample_data.csv
"""
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import streamlit as st

from utils import RATING_COLS, TARGET_COL, assign_clusters

BASE_DIR = Path(__file__).parent
MODEL_DIR = BASE_DIR / "model"

st.set_page_config(page_title="Airline Passenger Segmentation", page_icon="✈️", layout="wide")


@st.cache_resource
def load_artifacts():
    """Load the saved model files once and reuse them."""
    kmeans = joblib.load(MODEL_DIR / "kmeans.joblib")
    scaler = joblib.load(MODEL_DIR / "scaler.joblib")
    pca = joblib.load(MODEL_DIR / "pca.joblib")
    with open(MODEL_DIR / "meta.json", encoding="utf-8") as f:
        meta = json.load(f)
    profile = pd.read_csv(MODEL_DIR / "cluster_profile.csv")
    k_table = pd.read_csv(MODEL_DIR / "k_selection.csv")
    return kmeans, scaler, pca, meta, profile, k_table


kmeans, scaler, pca, meta, profile, k_table = load_artifacts()
COLORS = plt.get_cmap("tab10")


def pca_scatter(X_scaled, labels, max_points=5000):
    """2D PCA picture of the clusters (sampled when the data is large)."""
    idx = np.arange(len(labels))
    if len(idx) > max_points:
        idx = np.random.RandomState(42).choice(idx, max_points, replace=False)
    coords = pca.transform(X_scaled[idx])
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for c in sorted(set(labels)):
        mask = labels[idx] == c
        ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.6,
                   color=COLORS(int(c) % 10), label=f"Cluster {c}")
    ax.set_xlabel("PCA component 1")
    ax.set_ylabel("PCA component 2")
    ax.legend(markerscale=2)
    fig.tight_layout()
    return fig


def show_results(df, labels, X_scaled):
    results = df.copy()
    results["cluster"] = labels

    c1, c2 = st.columns(2)
    c1.metric("Passengers", f"{len(results):,}")
    c2.metric("Clusters found", results["cluster"].nunique())

    left, right = st.columns(2)
    with left:
        st.subheader("Cluster sizes")
        st.bar_chart(results["cluster"].value_counts().sort_index())
    with right:
        st.subheader("Clusters in 2D (PCA)")
        fig = pca_scatter(X_scaled, labels)
        st.pyplot(fig)
        plt.close(fig)

    st.subheader("Average rating per cluster")
    summary = results.groupby("cluster")[RATING_COLS].mean().round(2)
    summary.insert(0, "passengers", results["cluster"].value_counts().sort_index())
    if TARGET_COL in results.columns:
        summary.insert(1, "satisfied %", (results[TARGET_COL].eq("satisfied")
                                          .groupby(results["cluster"]).mean() * 100).round(1))
    st.dataframe(summary)

    st.subheader("What defines each cluster")
    st.dataframe(pd.DataFrame({"Cluster": list(meta["traits"].keys()),
                               "Main traits": list(meta["traits"].values())}))

    st.download_button("Download data with cluster column",
                       results.to_csv(index=False).encode("utf-8"),
                       file_name="passengers_with_clusters.csv", mime="text/csv")


# ---------- sidebar ----------
with st.sidebar:
    st.header("About")
    st.write(
        "Passengers are grouped with **K-Means** using their 14 service ratings "
        "(seat comfort, wifi, cleanliness, ...). Ratings are standardized "
        "(z-score) before clustering."
    )
    st.write(f"Number of clusters: **{meta['k']}**")
    st.caption("Satisfaction is not used for clustering. It is only used to describe the clusters.")

st.title("Airline Passenger Segmentation")
st.caption("Group airline passengers by how they rate the service, using K-Means clustering.")

tab1, tab2, tab3 = st.tabs(["Segment a CSV", "Single passenger", "Model details"])

# ---------- tab 1: CSV ----------
with tab1:
    source = st.radio("Data source", ["Use sample data", "Upload CSV"], horizontal=True)
    data = None
    if source == "Use sample data":
        data = pd.read_csv(BASE_DIR / "sample_data.csv")
    else:
        uploaded = st.file_uploader("Upload passenger CSV (needs the 14 rating columns)", type=["csv"])
        if uploaded is not None:
            data = pd.read_csv(uploaded)
        else:
            st.info("Upload a CSV with the 14 rating columns, or switch to the sample data.")
    if data is not None:
        st.write(f"Rows: {data.shape[0]}, Columns: {data.shape[1]}")
        try:
            labels, X_scaled = assign_clusters(data, scaler, kmeans)
        except ValueError as err:
            st.error(str(err))
        else:
            show_results(data, labels, X_scaled)

# ---------- tab 2: one passenger ----------
with tab2:
    st.write("Rate each service from 0 to 5 (0 = not applicable) and see which cluster the passenger belongs to.")
    with st.form("passenger_form"):
        cols = st.columns(3)
        ratings = {}
        for i, name in enumerate(RATING_COLS):
            ratings[name] = cols[i % 3].slider(name, 0, 5, 3)
        submitted = st.form_submit_button("Find cluster")
    if submitted:
        row = pd.DataFrame([ratings])
        label, _ = assign_clusters(row, scaler, kmeans)
        label = int(label[0])
        info = profile[profile["cluster"] == label].iloc[0]
        st.success(f"This passenger belongs to **Cluster {label}**")
        st.write(meta["traits"][str(label)])
        st.write(f"In this cluster, **{info['satisfied_pct']:.1f}%** of passengers are satisfied "
                 f"({int(info['size']):,} passengers, {info['share_pct']:.1f}% of all).")
        compare = pd.DataFrame({"This passenger": pd.Series(ratings),
                                f"Cluster {label} average": info[RATING_COLS].astype(float)})
        st.bar_chart(compare)

# ---------- tab 3: model details ----------
with tab3:
    st.subheader("Choosing the number of clusters")
    st.dataframe(k_table.round(4))
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(k_table["k"], k_table["inertia"], marker="o")
    axes[0].axvline(meta["k"], color="red", linestyle="--")
    axes[0].set_title("Elbow plot (inertia)")
    axes[0].set_xlabel("k")
    sil = k_table.dropna(subset=["silhouette"])
    axes[1].plot(sil["k"], sil["silhouette"], marker="o")
    axes[1].axvline(meta["k"], color="red", linestyle="--")
    axes[1].set_title("Silhouette score")
    axes[1].set_xlabel("k")
    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)
    st.caption(f"Red line = chosen k ({meta['k']}).")

    st.subheader("Cluster profile (full training data)")
    st.dataframe(profile)

    st.subheader("Cluster centers (standardized ratings)")
    centers = pd.DataFrame(kmeans.cluster_centers_, columns=RATING_COLS)
    centers.index.name = "cluster"
    fig, ax = plt.subplots(figsize=(10, 3 + 0.5 * len(centers)))
    sns.heatmap(centers, annot=True, fmt=".1f", cmap="RdBu_r", center=0, ax=ax)
    fig.tight_layout()
    st.pyplot(fig)
    plt.close(fig)
    st.caption("Positive = rated above the average passenger, negative = below.")

    with st.expander("Why are the ratings scaled?"):
        st.write(
            "K-Means uses distances, so every feature must be on a comparable scale. "
            "This project uses **StandardScaler (z-score)**: z = (x - mean) / std, "
            "which gives every rating mean 0 and standard deviation 1."
        )

## 12. Run the app

Open a terminal in this folder and run:

```
streamlit run app.py
```

The app opens at `http://localhost:8501`.

**Deploying on Streamlit Community Cloud**

1. Push `app.py`, `utils.py`, `requirements.txt`, `sample_data.csv` and the whole `model/` folder to GitHub.
2. Pin your `scikit-learn` version in `requirements.txt` (check with `pip show scikit-learn`), because the saved model files must be loaded with the same version.
3. On share.streamlit.io choose the repo, branch `main` and `app.py`, then deploy.